# Data Acquisition I — APIs

## Learning goals

By the end of this lesson, you should be able to:

- explain when an API is useful for obtaining external data;
- identify the main parts of an API request;
- read API documentation before writing code;
- make a `GET` request with Python using `requests`;
- inspect status codes and response data;
- work with JSON responses;
- send query parameters and headers;
- distinguish between authenticated and non-authenticated APIs;
- understand that APIs may be free, paid, or have several pricing tiers;
- store API credentials safely using environment variables.

The main idea of this lesson is:

> **We need external data. Is there a structured way to request it?**

## 1. We need external data

Until now, most of our datasets have already been available as files such as CSVs.

In real projects, the data we need may live somewhere else.

A useful decision process is:

```text
We need external data
        │
        ▼
Is it already available
as a downloadable dataset?
     │             │
    YES            NO
     │             │
     ▼             ▼
Download it    Is there an API?
                  │
                 YES
                  │
                  ▼
                 API
```

An **API** can provide a structured way for one application to request data from another system.

Later, we will consider what to do when the information exists on a website but no suitable API is available.

## 2. Web communication — only what we need

When we request data from an API, two systems communicate over the web.

```text
CLIENT                         SERVER
Python program  ── request ──► API
                ◄─ response ──
```

For this lesson, the important concepts are:

- **HTTP** → protocol used for communication;
- **URL** → address of a resource;
- **endpoint** → a specific API service/resource;
- **request** → what the client asks for;
- **response** → what the server returns.

We will focus only on **GET requests**, which are used to retrieve information.

### HTTP status codes

The response includes a status code.

Some common examples:

| Code | Meaning |
|---|---|
| `200` | OK |
| `400` | Bad Request |
| `401` | Unauthorized |
| `403` | Forbidden |
| `404` | Not Found |
| `429` | Too Many Requests |
| `500` | Server Error |

A successful request is normally in the `2xx` range.

### 💡 Quick Check 1 — Complete the sentence

Complete the code comments using:

`client`, `server`, `request`, `response`

In [ ]:
# Our Python program is the ________
# The API runs on a ________
# Python sends a ________
# The API returns a ________

## 3. What is an API?

**API** means **Application Programming Interface**.

An API defines rules that allow software systems to communicate.

For data acquisition, a REST API commonly exposes several **endpoints**.

For example:

```text
https://api.example.com/products
https://api.example.com/customers
https://api.example.com/orders
```

Each endpoint may return different data or accept different options.

### Common HTTP methods

REST APIs can support several HTTP methods:

| Method | Typical purpose |
|---|---|
| `GET` | retrieve data |
| `POST` | create/send new data |
| `PUT` | replace or fully update an existing resource |
| `PATCH` | partially update an existing resource |
| `DELETE` | remove a resource |

In this lesson we will only **use `GET`**, because our goal is to retrieve external data.

> `UPDATE` is not a standard HTTP method. Updates are usually performed with `PUT` or `PATCH`.

## 4. There is no universal API

This is one of the most important ideas in this lesson:

> **Different APIs have different endpoints, parameters, authentication methods, limits and pricing rules.**

There is no Python command that can tell you how every API works.

Before writing code, read the API documentation.

Look for:

1. **Base URL**
2. **Available endpoints**
3. **HTTP method**
4. **Required and optional parameters**
5. **Authentication**
6. **Response format**
7. **Rate limits**
8. **Pricing / usage restrictions**

The documentation is the source of truth for that API.

### APIs can differ in several ways

For example:

| API type | Possible example |
|---|---|
| No authentication + free | Public educational/open API |
| Authentication + free tier | API key with limited development quota |
| Authentication + paid | Commercial API |
| Free development + paid production | Many commercial services |

These categories are independent.

An API being **free** does not mean that it requires **no authentication**.

An API requiring authentication does not necessarily mean that it is **paid**.

In this lesson we will use only two examples:

### Where the ISS At

- public REST API;
- no authentication currently required;
- returns JSON;
- rate limited.

Documentation:

https://wheretheiss.at/w/developer

### NewsAPI

- requires an API key;
- offers a free development plan;
- commercial/production usage requires a paid plan.

Documentation:

https://newsapi.org/docs

Getting started / obtaining an API key:

https://newsapi.org/docs/get-started

Registration page:

https://newsapi.org/register

This gives us enough examples to understand the concepts without learning several unrelated APIs.

## 5. Required libraries

This lesson uses:

- `requests` → send HTTP requests;
- `python-dotenv` → load credentials from a `.env` file.

Anaconda Distribution includes many common packages, but the exact package set can vary by version and environment.

Before installing anything, try:

```python
import requests
from dotenv import load_dotenv
```

If an import fails, install the missing package with **conda** rather than mixing package managers in the same environment.

### macOS / Linux

1. Open a new **Terminal**.
2. Copy and paste the command for the missing library.
3. Press Enter.

For `requests`:

```bash
conda install -y anaconda::requests
```

For `python-dotenv`:

```bash
conda install -y anaconda::python-dotenv
```

The `-y` option automatically answers **yes** to the installation confirmation.

### Windows

1. Open a new **Anaconda PowerShell Prompt**.
2. Copy and paste the command for the missing library.
3. Press Enter.

For `requests`:

```powershell
conda install -y anaconda::requests
```

For `python-dotenv`:

```powershell
conda install -y anaconda::python-dotenv
```

Official package pages:

https://anaconda.org/anaconda/requests

https://anaconda.org/anaconda/python-dotenv

> Avoid installing packages into the `base` environment unless that is the environment intentionally used for the course.

## 6. Making a GET request with `requests`

Python's `requests` library makes HTTP requests easier to work with.

We will start with **Where the ISS At**, because it does not require credentials.

Before using a satellite ID, let's follow the API documentation and begin with the general endpoint:

```text
https://api.wheretheiss.at/v1/satellites
```

This endpoint returns the satellites currently available through the API.

Documentation:

https://wheretheiss.at/w/developer

In [ ]:
import requests

satellites_url = "https://api.wheretheiss.at/v1/satellites"

response = requests.get(satellites_url)

print(response.status_code)

The `response` object contains much more than the data itself.

Useful attributes and methods include:

- `response.status_code`
- `response.headers`
- `response.text`
- `response.json()`

In [ ]:
print("Status:", response.status_code)
print("Content type:", response.headers.get("Content-Type"))

### Discovering the satellite ID

Now inspect the JSON returned by `/satellites`:


In [ ]:
satellites = response.json()

display(satellites)

The API currently lists the International Space Station with the NORAD catalog ID `25544`.

Now that we have discovered the ID from the API, we can use the more specific endpoint:

```text
/satellites/{id}
```

For the ISS:

```text
https://api.wheretheiss.at/v1/satellites/25544
```


In [ ]:
iss_id = satellites[0]["id"]

iss_url = f"https://api.wheretheiss.at/v1/satellites/{iss_id}"

iss_response = requests.get(iss_url)
iss_response.raise_for_status()

print(iss_response.status_code)

### Checking the request

A request can fail even when the Python code itself is valid.

`raise_for_status()` raises an error for unsuccessful HTTP responses.

In [ ]:
response.raise_for_status()

### 💡 Quick Check 2 — Complete the code

Make a GET request and print its status code.

In [ ]:
satellites_url = "https://api.wheretheiss.at/v1/satellites"

response = requests._____(satellites_url)

print(response.__________)

## 7. JSON responses

APIs often return data as **JSON**.

JSON is useful because its structure maps naturally to Python dictionaries and lists.

`response.json()` converts the JSON response into Python objects.

In [ ]:
data = iss_response.json()

print(type(data))
display(data)

### Exploring JSON as an interactive tree

Small JSON responses are easy to read directly.

Larger or nested responses can be much harder to understand.

Jupyter can display JSON as an **interactive expandable tree** using `IPython.display.JSON`.

This is useful for answering questions such as:

- Which keys exist?
- Which values are dictionaries?
- Which values are lists?
- What path do I need to follow to reach a specific value?


In [ ]:
from IPython.display import JSON

JSON(data)

You can also display the tree expanded:

```python
JSON(data, expanded=True)
```

The visual structure maps directly to Python objects:

```text
JSON object  → Python dictionary
JSON array   → Python list
key          → dictionary key
nested item  → nested dictionary/list
```

For example, a structure such as:

```text
root
└── articles
    └── 0
        ├── source
        │   └── name
        └── title
```

would be accessed in Python as:

```python
data["articles"][0]["source"]["name"]
```

This makes the JSON tree a useful tool for discovering **how to navigate nested API responses**.


### Optional external tool

If you ever need to inspect JSON outside Jupyter, a visual tool such as **JSON Crack** can also help:

https://jsoncrack.com/

> Never paste API keys, tokens, credentials, personal data, or other sensitive information into an external website.


Now we can access values just as we would in a Python dictionary.

In [ ]:
print("Latitude:", data["latitude"])
print("Longitude:", data["longitude"])
print("Altitude:", data["altitude"])
print("Velocity:", data["velocity"])

And if the structure is useful for analysis, we can create a Pandas DataFrame.

In [ ]:
import pandas as pd

iss_position = pd.DataFrame([{
    "latitude": data["latitude"],
    "longitude": data["longitude"],
    "altitude": data["altitude"],
    "velocity": data["velocity"]
}])

display(iss_position)

### 💡 Quick Check 3 — Complete the code

Extract the ISS `visibility` and `timestamp` values from the JSON response.

In [ ]:
visibility = data[________]
timestamp = data[________]

print(visibility)
print(timestamp)

## 8. Parameters

Endpoints often accept **parameters** that modify the request.

A common type is a **query parameter**.

Instead of manually constructing this:

```text
...?units=miles
```

we can pass a dictionary to `requests`.

In [ ]:
params = {
    "units": "miles"
}

response = requests.get(
    iss_url,
    params=params
)

response.raise_for_status()

data_miles = response.json()

print("Altitude:", data_miles["altitude"])
print("Units:", data_miles["units"])

Using a dictionary is usually clearer and safer than manually concatenating strings into the URL.

Again, the API documentation tells us:

- which parameters exist;
- which are required;
- which values are accepted.

### 💡 Quick Check 4 — Complete the code

Send `units=miles` as a query parameter.

In [ ]:
params = {
    "units": ________
}

response = requests.get(
    iss_url,
    params=________
)

response.raise_for_status()

## 9. Authentication

Some APIs need to know who is making the request.

Common authentication mechanisms include:

- API keys;
- tokens;
- OAuth;
- client ID + client secret.

The exact mechanism depends on the API.

**Always read the documentation.**

For this lesson, we only need to understand the simplest case: an API key.

### Example: NewsAPI

NewsAPI requires an API key.

Its documentation allows the key to be sent in several ways, including the `X-Api-Key` header.

A header is additional information attached to the HTTP request.

Conceptually:

```text
GET /v2/everything
Headers:
    X-Api-Key: <your secret key>
Parameters:
    q: artificial intelligence
```

We should **not** place the real key directly in the notebook.

## 10. Never hard-code credentials

This is unsafe:

```python
API_KEY = "my-real-secret-api-key"
```

If the notebook is uploaded to GitHub, the secret is now part of the repository.

Instead:

```text
.env
  │
  ├── contains secrets
  │
  └── MUST be ignored by Git

Notebook
  │
  ├── may READ secrets
  │
  └── must NEVER DISPLAY secrets
```

### Step 1 — Create a `.env` file

Create a **plain-text file** named exactly:

```text
.env
```

The leading dot is part of the filename. Do not call it `env.txt` or `.env.txt`.

You can create it with a plain-text editor such as **Sublime Text**:

https://www.sublimetext.com/download

Save the file in the **same project folder as your notebook**.

Example:

```text
my_api_project/
│
├── analysis.ipynb
├── .env
└── .gitignore
```

Inside `.env`, write one environment variable per line:

```text
NEWS_API_KEY=your-real-api-key
```

Do not add Python syntax such as:

```python
NEWS_API_KEY = "..."
```

The `.env` file is not Python code. It is a simple key-value configuration file.

### Step 2 — Add `.env` to `.gitignore`

Your `.gitignore` should contain:

```text
.env
```

This prevents Git from adding a new `.env` file to the repository.

⚠️ **Important:** `.gitignore` does not stop tracking a file that was already committed.

If a secret has already been pushed to GitHub, assume it has been exposed and **revoke / regenerate it**.

### Step 3 — Load the variable with `python-dotenv`

If `python-dotenv` is not available in your environment, install it first.

Then:

In [ ]:
from dotenv import load_dotenv
import os

# Search for a .env file in the notebook/project directory
# and load its key-value pairs into environment variables.
load_dotenv()

# Read the NEWS_API_KEY environment variable loaded from .env.
news_api_key = os.getenv("NEWS_API_KEY")

Notice what we do **not** do:

```python
print(news_api_key)
news_api_key
```

A Jupyter notebook is a JSON file and stores cell outputs.

If you display a secret, the value can be saved inside the `.ipynb` file even though the `.env` file itself is ignored.

### Secrets can leak indirectly

This can also expose a credential:

```python
headers = {
    "X-Api-Key": news_api_key
}

headers
```

The last expression would be displayed and stored as notebook output.

Errors, debug messages and logs can also reveal sensitive information.

So the rule is broader than:

> Don't print the API key.

The real rule is:

> **Never display secrets or objects that contain secrets.**

### Before committing a notebook that uses credentials

Use this checklist:

```text
1. Add .env to .gitignore
2. Create a commit containing only the updated .gitignore
3. Store the secret in .env
4. Verify that secrets are never displayed
5. Verify that objects containing secrets are never displayed
6. Restart the kernel and clear all outputs
7. Save the notebook
8. Run git status
9. Verify .env is NOT staged
10. Commit / push
```

Committing `.gitignore` first establishes the protection before the secret file enters your normal workflow.

If a credential has ever been committed or pushed, removing it later is **not enough**.

Revoke or regenerate the credential.

## 11. Using an authenticated API safely

A request to NewsAPI could look like this:

In [ ]:
news_url = "https://newsapi.org/v2/everything"

params = {
    "q": "data science",
    "language": "en",
    "pageSize": 5
}

headers = {
    "X-Api-Key": news_api_key
}

# Do not display `headers`

# Uncomment only after creating your own API key:
#
# response = requests.get(
#     news_url,
#     params=params,
#     headers=headers
# )
#
# response.raise_for_status()
# news_data = iss_response.json()

The important lesson is not the NewsAPI syntax itself.

Another authenticated API may use:

```text
Authorization: Bearer <token>
```

or query parameters, OAuth, client credentials, or another mechanism.

**Do not copy authentication code from one API to another without checking the documentation.**

### 💡 Quick Check 5 — Complete the security checklist

Which file should contain the secret, and which file should tell Git to ignore it?

In [ ]:
# File that stores credentials:
secret_file = "________"

# File that prevents Git from tracking it:
ignore_file = "________"

print(secret_file)
print(ignore_file)

## 12. A practical API workflow

When using an unfamiliar API, follow this process:

```text
1. Define the data you need
        ↓
2. Find the API documentation
        ↓
3. Identify the endpoint
        ↓
4. Check authentication
        ↓
5. Check pricing / rate limits
        ↓
6. Identify parameters
        ↓
7. Make the request
        ↓
8. Check status code
        ↓
9. Parse the response
        ↓
10. Inspect the data structure
        ↓
11. Transform into useful data
```

Do not start by randomly changing URLs until something works.

**Read the documentation first.**

## 13. Final Challenge — Complete the request

Use the public ISS API to:

1. request the list of available satellites;
2. extract the first satellite ID;
3. build the position endpoint using that ID;
4. request the current ISS information;
5. verify the request succeeded;
6. convert the JSON response to Python;
7. build a one-row DataFrame containing:
   - latitude;
   - longitude;
   - altitude;
   - visibility.

In [ ]:
satellites_url = "https://api.wheretheiss.at/v1/satellites"

response = requests._____(satellites_url)
response.________________()

satellites = response.________()
iss_id = satellites[0][________]

iss_url = f"https://api.wheretheiss.at/v1/satellites/{iss_id}"

response = requests._____(iss_url)
response.________________()

iss_data = response.________()

result = pd.DataFrame([{
    "latitude": iss_data[________],
    "longitude": iss_data[________],
    "altitude": iss_data[________],
    "visibility": iss_data[________]
}])

display(result)

## Summary

Today you learned that APIs are **structured interfaces for obtaining external data**.

The core workflow is:

```text
Documentation
    ↓
Endpoint + parameters + authentication
    ↓
HTTP GET request
    ↓
Status code
    ↓
JSON response
    ↓
Python objects
    ↓
DataFrame / analysis
```

You also learned that:

- APIs are not all the same;
- some APIs require authentication and others do not;
- authentication and pricing are separate concepts;
- free APIs may still have quotas and restrictions;
- paid APIs may offer free development tiers;
- API documentation must be read before writing code;
- credentials should never be hard-coded or displayed;
- `.env` + `.gitignore` protects the credential file;
- notebook outputs must also be checked before publishing.

### Further exploration

Later you may encounter:

- POST requests;
- OAuth;
- pagination;
- API wrappers / SDKs;
- more complex authentication flows;
- retry logic;
- rate-limit handling.

These are useful tools, but they are not required to understand the fundamentals of consuming an API.